In [20]:
from debyecalculator import DebyeCalculator

import numpy as np
from ase.io import read

import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.colors import ListedColormap
from matplotlib.ticker import FormatStrFormatter

In [54]:
dirs = range(500, 1050, 50)

atoms_list = []

for i in dirs:
    atoms = read(f"../data/PDFs/selected_frames_{i}.xyz", index=':')
    atoms_list.append(atoms)
    print(len(atoms))

In [55]:
calc = DebyeCalculator(
    qmin=0.7,        # Minimum Q value (1/Å).
    qmax=40,       # Maximum Q value (1/Å).
    qstep=0.09,       # Step size for Q (1/Å), Nyquist-sampled by default.
    qdamp=0.0343,       # Dampening factor.

    rmin=0,        # Minimum R value (Å).
    rmax=10.0,       # Maximum R value (Å).
    rstep=0.01,      # Step size for R (Å).
    rthres=0.0,      # Lower threshold for radial distance (Å) in G(r) calculations.

    biso=0.2,        # Isotropic atomic displacement factor (Debye-Waller factor).
    device='cpu',   # Compute on 'cuda' for GPU or 'cpu' for CPU.
    batch_size=None, # Batch size for pairwise distance calculations (auto-determined if not set).
    lorch_mod=False, # Apply Lorch modification factor in G(r) calculations.
    rad_type='neutron', # Type of radiation for scattering: 'xray'/'x' or 'neutron'/'n'.
    profile=False,   # Enable profiling for performance analysis.
)

In [56]:
r_list = []
gr_list = []

for j in atoms_list:
    r_list_j = []
    gr_list_j = []

    for i in j:
        r, gr = calc.gr(i)
        r_list_j.append(r)
        gr_list_j.append(gr)

    r_list.append(r_list_j)
    gr_list.append(gr_list_j)

In [57]:
r_mean_list = []
gr_mean_list = []
gr_error_list = []

for k, l in zip(r_list, gr_list):
    r_mean = np.mean(k, axis=0)
    gr_mean = np.mean(l, axis=0)
    gr_error = np.std(l, axis=0)

    r_mean_list.append(r_mean)
    gr_mean_list.append(gr_mean)
    gr_error_list.append(gr_error)

In [316]:
fig, ax = plt.subplots(figsize=(8, 6))

top = mpl.colormaps['Oranges'].resampled(130)
bottom = mpl.colormaps['Greens'].resampled(130)

newcolors = np.vstack((top(np.linspace(1, 0.3, 72)),
                       bottom(np.linspace(0.3, 1, 128))))
newcmp = ListedColormap(newcolors, name='OrangeGreen')

colormap = newcmp
colors = colormap(np.linspace(0, 1, 11))

for idx, (r_means, gr_means, gr_errors, temp) in enumerate(zip(r_mean_list, gr_mean_list, gr_error_list, dirs)):

    ax.plot(r_means, gr_means, color=colors[idx], lw=0.5, label=f"{temp}K")
    ax.fill_between(r_means, gr_means - gr_errors, gr_means + gr_errors, color=colors[idx], alpha=0.2)

    ax.set(xlabel='r ($Å$)', ylabel='G(r) (a.u.)', yticks=[])
    ax.set_xlim(0, 10)

norm = mpl.colors.Normalize(vmin=500, vmax=1000)
sm = mpl.cm.ScalarMappable(norm=norm, cmap=colormap)
sm.set_array([])
cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
fig.colorbar(sm, cax=cbar_ax, orientation='vertical', label='Temperature (K)', ticks=[500, 680, 1000])

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(3.35, 2.53))

colors = ['#F39200', '#F39200', '#F39200', '#0D6A38', '#0D6A38', '#0D6A38', '#0D6A38', '#0D6A38', '#0D6A38', '#0D6A38', '#0D6A38']

for idx, (r_means, gr_means, gr_errors, temp) in enumerate(zip(r_mean_list, gr_mean_list, gr_error_list, dirs)):

    ax.plot(r_means, gr_means, color=colors[idx], lw=0.8, label=f"{temp}K")
    ax.set(ylabel='$G(r)$')
    ax.set_xlim(0, 10)

ax.tick_params(axis='both', direction='in', length=4, width=1.3, top=True, bottom=True, left=True, right=True)

ax.set_yticks([-3.0, -2.0, -1.0, 0, 1.0, 2.0, 3.0])
ax.yaxis.set_major_formatter(FormatStrFormatter('%.1f'))
ax.set_xticklabels([])

plt.savefig('neutron_v4.png', bbox_inches='tight')
plt.show()

In [198]:
calc_x = DebyeCalculator(
    qmin=0.7,        # Minimum Q value (1/Å).
    qmax=40,       # Maximum Q value (1/Å).
    qstep=0.09,       # Step size for Q (1/Å), Nyquist-sampled by default.
    qdamp=0.0486,       # Dampening factor.

    rmin=0,        # Minimum R value (Å).
    rmax=10.0,       # Maximum R value (Å).
    rstep=0.01,      # Step size for R (Å).
    rthres=0.0,      # Lower threshold for radial distance (Å) in G(r) calculations.

    biso=0.2,        # Isotropic atomic displacement factor (Debye-Waller factor).
    device='cpu',   # Compute on 'cuda' for GPU or 'cpu' for CPU.
    batch_size=None, # Batch size for pairwise distance calculations (auto-determined if not set).
    lorch_mod=False, # Apply Lorch modification factor in G(r) calculations.
    rad_type='xray', # Type of radiation for scattering: 'xray'/'x' or 'neutron'/'n'.
    profile=False,   # Enable profiling for performance analysis.
)

In [199]:
r_list_x = []
gr_list_x = []

for j in atoms_list:
    r_list_j = []
    gr_list_j = []

    for i in j:
        r, gr = calc_x.gr(i)
        r_list_j.append(r)
        gr_list_j.append(gr)

    r_list_x.append(r_list_j)
    gr_list_x.append(gr_list_j)

In [209]:
r_list_x = r_list[11:]
gr_list_x = gr_list[11:]

In [211]:
r_mean_list_x = []
gr_mean_list_x = []
gr_error_list_x = []

for k, l in zip(r_list_x, gr_list_x):
    r_mean = np.mean(k, axis=0)
    gr_mean = np.mean(l, axis=0)
    gr_error = np.std(l, axis=0)

    r_mean_list_x.append(r_mean)
    gr_mean_list_x.append(gr_mean)
    gr_error_list_x.append(gr_error)

In [312]:
fig, ax = plt.subplots(figsize=(8, 6))

top = mpl.colormaps['Oranges'].resampled(130)
bottom = mpl.colormaps['Greens'].resampled(130)

newcolors = np.vstack((top(np.linspace(1, 0.3, 72)),
                       bottom(np.linspace(0.3, 1, 128))))
newcmp = ListedColormap(newcolors, name='OrangeGreen')

colormap = newcmp
colors = colormap(np.linspace(0, 1, 11))

for idx, (r_means, gr_means, gr_errors, temp) in enumerate(zip(r_mean_list_x, gr_mean_list_x, gr_error_list_x, dirs)):

    ax.plot(r_means, gr_means, color=colors[idx], lw=0.5, label=f"{temp}K")
    ax.fill_between(r_means, gr_means - gr_errors, gr_means + gr_errors, color=colors[idx], alpha=0.2)

    ax.set(xlabel='r ($Å$)', ylabel='G(r) (a.u.)', yticks=[])
    ax.set_xlim(0, 10)

norm = mpl.colors.Normalize(vmin=500, vmax=1000)
sm = mpl.cm.ScalarMappable(norm=norm, cmap=colormap)
sm.set_array([])
cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
fig.colorbar(sm, cax=cbar_ax, orientation='vertical', label='Temperature (K)', ticks=[500, 680, 1000])

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6), dpi=300)

colors = ['#F39200', '#F39200', '#F39200', '#0D6A38', '#0D6A38', '#0D6A38', '#0D6A38', '#0D6A38', '#0D6A38', '#0D6A38', '#0D6A38']

for idx, (r_means, gr_means, gr_errors, temp) in enumerate(zip(r_mean_list_x, gr_mean_list_x, gr_error_list_x, dirs)):

    ax.plot(r_means, gr_means, color=colors[idx], lw=0.5, label=f"{temp}K")
    ax.fill_between(r_means, gr_means - gr_errors, gr_means + gr_errors, color=colors[idx], alpha=0.2)

    ax.set(xlabel='$r$ ($\mathrm{Å}$)', ylabel='$G(r)$')
    ax.set_xlim(0, 10)

ax.tick_params(axis='both', direction='in', length=7, width=2, top=True, bottom=True, left=True, right=True)
ax.get_xticks()
ax.get_yticks()

ax.yaxis.set_major_formatter(FormatStrFormatter('%.1f'))

for spine in ax.spines.values():
    spine.set_linewidth(2.5)

plt.savefig('xray_v5.png', bbox_inches='tight')
plt.show()